In [1]:
!pip install multilingual-clip

In [63]:
import matplotlib.image as mpimg
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import json
import torch
from collections import Counter
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import clip
#from google.colab import drive
import os
from os import listdir
from os.path import isfile, join
import matplotlib.image as mpimg
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import json
import torch
from collections import Counter
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import clip
#from google.colab import drive
import os
from os import listdir
from os.path import isfile, join

In [64]:
def find_most_common_answer(answers):
    answer_counter = Counter(answers)
    most_common_answers = answer_counter.most_common()
    most_common_answer, count = most_common_answers[0]
    return most_common_answer

def select_most_common_answers(df):
    selected_answers = []
    for idx, row in df.iterrows():
        answers = [answer["answer"] for answer in row["answers"]]
        selected_answer = find_most_common_answer(answers)

        selected_answers.append({"answer": selected_answer
        })

    # Update the "answer" and "answer_confidence" columns in the DataFrame
    df[["answer"]] = pd.DataFrame(selected_answers)

    return df.drop(["answers"], axis=1)

def plot_loss(train_loss, val_loss):
    epochs = range(1, len(train_loss) + 1)

    plt.plot(epochs, train_loss, label='Training Loss')
    plt.plot(epochs, val_loss, label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.show()

def plot_img(path):
    # Load the JPEG image
    image = mpimg.imread(path)

    # Plot the image
    plt.imshow(image)
    plt.axis('off')  # Remove axis ticks and labels
    plt.show()

In [65]:
def dataloader_json(path,test=False):
    # Load the JSON file
    with open(path, 'r') as f:
        data = json.load(f)
    # Create a DataFrame from the loaded JSON data
    df = pd.DataFrame(data)

    if test:
        return df

    return select_most_common_answers(df)

In [66]:
train_df = dataloader_json("Annotations/Annotations/train_beng.json")

train_df

,image,question,answer_type,answerable,answer
0,VizWiz_train_00000000.jpg,এই পণ্যের নাম কি?,other,1,বেসিল পাতা
1,VizWiz_train_00000001.jpg,তুমি কি বলতে পারবে এর মধ্যে কি আছে দয়া করে?,other,1,কোকো
2,VizWiz_train_00000002.jpg,এটা কি সস নাকি টমেটো? ধন্যবাদ.,other,1,টমেটো
3,VizWiz_train_00000003.jpg,এই ছবিতে ক্যাপচা কি?,other,1,ছাকন
4,VizWiz_train_00000004.jpg,এসব কি?,other,1,সৌরজগত্‍
...,...,...,...,...,...
20518,VizWiz_train_00023949.jpg,ল্যাপটপের রং কি?,other,1,কালো
20519,VizWiz_train_00023950.jpg,"তুমি এটা দেখতে পাচ্ছ? যদি তাই হয়, তাহলে বলো এ...",other,1,অ‌ধাতু
20520,VizWiz_train_00023951.jpg,আমাকে তিনটা সংখ্যা লিখতে হবে কেন?,unanswerable,0,প্রযোজ্য নয় এমন ছবি
20521,VizWiz_train_00023952.jpg,এটা কি বক্স?,yes/no,1,না


In [67]:
train_df['answer'].value_counts()

answer
অ‌ধাতু                         3904
প্রযোজ্য নয়                   3022
না                              569
হ্যাঁ                           519
সাদা                            296
                               ... 
আকার পরিবর্তন                     1
জালপেনো অংশগুলির পূর্বদৃশ্য       1
মিনিট ৩০ সেকেন্ড                  1
রাস্পবেরি শাওয়ারেজ               1
নতুন ippui                        1
Name: count, Length: 4899, dtype: int64

In [68]:
val_df = dataloader_json("Annotations/Annotations/val_beng.json")
val_df

,image,question,answer_type,answerable,answer
0,VizWiz_val_00000000.jpg,"ঠিক আছে। অন্য একটি ছবি আছে, আশা করি এটাই ভালো ...",unanswerable,0,অ‌ধাতু
1,VizWiz_val_00000001.jpg,আপনি কি আমাকে বলবেন এই ওষুধটা কিসের?,other,1,রাতের সময়
2,VizWiz_val_00000002.jpg,এই বইয়ের শিরোনাম কী?,other,1,বছর
3,VizWiz_val_00000003.jpg,নীলটা কোনটা?,other,1,ডানদিকে
4,VizWiz_val_00000004.jpg,তীর কি বলে?,unanswerable,0,অ‌ধাতু
...,...,...,...,...,...
4314,VizWiz_val_00004314.jpg,এটা কি?,other,1,সালাদের পোশাক
4315,VizWiz_val_00004315.jpg,এটা কি আধুনিক?,unanswerable,0,প্রযোজ্য নয়
4316,VizWiz_val_00004316.jpg,"আমি এই ব্যাটারিটা কিনব, তুমি কি মডেল আর নাম দে...",unanswerable,0,অ‌ধাতু
4317,VizWiz_val_00004317.jpg,এটা কি ধরনের মিশ্রণ?,other,1,কেক


In [69]:
data_df = pd.concat((train_df,val_df), axis =0,ignore_index=True)
data_df

,image,question,answer_type,answerable,answer
0,VizWiz_train_00000000.jpg,এই পণ্যের নাম কি?,other,1,বেসিল পাতা
1,VizWiz_train_00000001.jpg,তুমি কি বলতে পারবে এর মধ্যে কি আছে দয়া করে?,other,1,কোকো
2,VizWiz_train_00000002.jpg,এটা কি সস নাকি টমেটো? ধন্যবাদ.,other,1,টমেটো
3,VizWiz_train_00000003.jpg,এই ছবিতে ক্যাপচা কি?,other,1,ছাকন
4,VizWiz_train_00000004.jpg,এসব কি?,other,1,সৌরজগত্‍
...,...,...,...,...,...
24837,VizWiz_val_00004314.jpg,এটা কি?,other,1,সালাদের পোশাক
24838,VizWiz_val_00004315.jpg,এটা কি আধুনিক?,unanswerable,0,প্রযোজ্য নয়
24839,VizWiz_val_00004316.jpg,"আমি এই ব্যাটারিটা কিনব, তুমি কি মডেল আর নাম দে...",unanswerable,0,অ‌ধাতু
24840,VizWiz_val_00004317.jpg,এটা কি ধরনের মিশ্রণ?,other,1,কেক


In [70]:
ans_lb = LabelEncoder()
data_df['answer'] = ans_lb.fit_transform(data_df['answer'])
ans_type_lb = LabelEncoder()
data_df['answer_type']= ans_type_lb.fit_transform(data_df['answer_type'])

In [71]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [72]:
from multilingual_clip import pt_multilingual_clip
import transformers

model_name = 'M-CLIP/XLM-Roberta-Large-Vit-L-14'

# Load Model & Tokenizer
model = pt_multilingual_clip.MultilingualCLIP.from_pretrained(model_name)
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)

#embeddings = model.forward(texts, tokenizer)
#print(embeddings.shape)

/home/arif/anaconda3/envs/env_ratnabali/lib/python3.9/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/arif/anaconda3/envs/env_ratnabali/lib/python3.9/site-packages/transformers/modeling_utils.py:442: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are

In [73]:
import torch
model_clip, preprocess = clip.load("ViT-L/14", device=device)

from torchvision import transforms
# Prepare the image
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
#model_clip.context_length=512
encodings = []
for img , question in tqdm(zip(data_df['image'],data_df['question'])):
    if "train" in img:
        image = preprocess(Image.open(f'train/{img}')).unsqueeze(0).to(device)
    elif "test" in img:
       image = preprocess(Image.open(f'test/{img}')).unsqueeze(0).to(device)
    else:
        image = preprocess(Image.open(f'val/{img}')).unsqueeze(0).to(device)

    text = tokenizer(question, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        image_encoding = model_clip.encode_image(image)
        text_encoding = model.forward(question, tokenizer).to(device)
        #print(text_encoding.shape)
        encodings.append(torch.cat([image_encoding, text_encoding], dim=-1))

5it [00:00, 21.15it/s]

torch.Size([1, 768])
torch.Size([1, 768])
torch.Size([1, 768])
torch.Size([1, 768])
torch.Size([1, 768])


8it [00:00, 20.32it/s]

torch.Size([1, 768])
torch.Size([1, 768])
torch.Size([1, 768])
torch.Size([1, 768])


14it [00:00, 20.06it/s]

...

68it [00:03, 23.01it/s]

torch.Size([1, 768])
torch.Size([1, 768])
torch.Size([1, 768])
torch.Size([1, 768])
torch.Size([1, 768])


73it [00:03, 22.27it/s]

torch.Size([1, 768])
torch.Size([1, 768])
torch.Size([1, 768])
torch.Size([1, 768])


KeyboardInterrupt: 

In [41]:
torch.save(encodings,"model-ml.pt")

In [61]:
encodings

[tensor([[0.0085, 1.1250, 0.0895,  ..., 0.1511, 0.2544, 0.4331]],
        device='cuda:0'),
 tensor([[ 0.1897,  0.3145,  0.0903,  ...,  0.0042,  0.2063, -0.0134]],
        device='cuda:0'),
 tensor([[ 0.6279, -0.0081, -0.1984,  ..., -0.3382,  0.3426, -0.1012]],
        device='cuda:0'),
 tensor([[0.4348, 0.6797, 0.3953,  ..., 0.0102, 0.2194, 0.1637]],
        device='cuda:0'),
 tensor([[ 1.0811,  0.6187, -0.0789,  ..., -0.0274,  0.0387, -0.1133]],
        device='cuda:0'),
 tensor([[-0.1033,  1.1816,  0.1733,  ..., -0.0281,  0.0918, -0.1423]],
        device='cuda:0'),
 tensor([[-0.5464,  0.8525,  0.0166,  ...,  0.0696,  0.5660,  0.6703]],
        device='cuda:0'),
 tensor([[ 0.0735,  0.9424, -0.2874,  ...,  0.3132,  0.3494, -0.3200]],
        device='cuda:0'),
 tensor([[ 0.0836,  0.5137,  0.0739,  ..., -0.1577,  0.4445, -0.1183]],
        device='cuda:0'),
 tensor([[-0.4580,  0.7593,  0.1772,  ..., -0.1968,  0.1190, -0.0096]],
        device='cuda:0'),
 tensor([[ 0.5298,  0.5601, -0.2

In [42]:
print(len(data_df))

24842


In [43]:
# Generate the indices for train-test split
indices = np.arange(len(data_df))

# Perform train-test split
train_indices, test_indices = train_test_split(indices, test_size=0.05, random_state=42)

In [44]:
train = data_df.iloc[train_indices] 
train = train.reset_index(drop=True)
train

,image,question,answer_type,answerable,answer
0,VizWiz_train_00017501.jpg,এর ভেতরে কি আছে?,2,0,349
1,VizWiz_val_00003817.jpg,এই সিসিটিভির ক্রমিক সংখ্যা কি?,2,0,349
2,VizWiz_train_00002858.jpg,শার্টটা কিসের?,1,1,3541
3,VizWiz_train_00004344.jpg,আমি কি দেখছি?,1,1,4229
4,VizWiz_train_00014612.jpg,আপনি কি দয়া করে আমাকে বলবেন যে এটা টিভি স্ট্য...,3,1,5120
...,...,...,...,...,...
23594,VizWiz_val_00001052.jpg,এটা কি বলে? এটা কি বলে?,2,0,3026
23595,VizWiz_train_00005390.jpg,এটা কি ধরনের সোডা?,1,1,2841
23596,VizWiz_train_00000860.jpg,বালম বি-এল-এম কি এই কন্টেইনার সম্পর্কে লিখেছে?,3,1,2514
23597,VizWiz_train_00015795.jpg,শার্টটা কি বলে?,1,1,428


In [45]:
test = data_df.iloc[test_indices]
test = test.reset_index(drop=True)
test

,image,question,answer_type,answerable,answer
0,VizWiz_train_00005163.jpg,এটা কি আলো নাকি অন্ধকার?,1,1,1502
1,VizWiz_train_00004569.jpg,থার্মোস্তা কি বলে?,2,0,3025
2,VizWiz_train_00023795.jpg,এসব কি? এটা কি?,1,1,4391
3,VizWiz_val_00000320.jpg,পর্দায় কি লেখা আছে?,1,1,1540
4,VizWiz_train_00013788.jpg,কে- মাইন,1,1,2152
...,...,...,...,...,...
1238,VizWiz_train_00003806.jpg,এই পৃষ্ঠা কী বলে?,2,0,3025
1239,VizWiz_train_00004115.jpg,এটা কিসের মডেল?,1,1,3719
1240,VizWiz_train_00004547.jpg,এই খাবারের নাম কি?,1,1,2545
1241,VizWiz_train_00019406.jpg,এই বাক্সে কি আছে?,2,0,349


In [46]:
embedding_size = encodings[0].shape[1] #change by Arif old was 768
classes = len(np.unique(ans_lb.classes_))
aux_classes = len(np.unique(train['answer_type']))
BATCH_SIZE = 64

In [53]:
encodings[0].shape[1]

1536

In [54]:
encd=torch.load('model-ml.pt')
class dataset(Dataset):
    def __init__(self, indices, answers, types, length):
        self.indices = indices
        self.answers = answers
        self.types = types
        self.length = length

    def __getitem__(self, index):
            if self.length <= 24842:
                return encd[self.indices[index]].float() , torch.tensor(int(self.answers[index])), torch.tensor(int(self.types[index]))
            
            return encd[self.indices[index]].float() , torch.tensor(int(self.answers[index % (self.length/2)])), torch.tensor(int(self.types[index % (self.length/2)]))
            
    def __len__(self):
          return self.length

/tmp/ipykernel_2294317/1329949184.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  encd=torch.load('model-ml.pt')


In [55]:

#trainDataset = dataset(np.concatenate((train_indices, train_indices_aug)), train['answer'], train['answer_type'], len(train_indices)*2)
trainDataset = dataset(np.array(range(train_indices.size)), train['answer'] , train['answer_type'], len(train_indices))

valDataset = dataset(np.array(range(test_indices.size)), test['answer'], test['answer_type'], len(test_indices))

#valDataset = dataset(np.array(list(range(encd.shape[0] - 5642, encd.shape[0]))), val['answer'], val['answer_type'], 4319)
#valDataset = dataset(np.array(list(range(encd.shape[0] - 5642, encd.shape[0]))), pd.DataFrame(e).iloc[24842:], val['answer_type'], 5642)

train_dataloader = DataLoader(trainDataset, batch_size = BATCH_SIZE, shuffle = True, num_workers=0)

val_dataloader = DataLoader(valDataset, batch_size = BATCH_SIZE, shuffle = True, num_workers=0)

In [56]:
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.ln1 = nn.LayerNorm(embedding_size*2)
        self.dp1 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(embedding_size* 2, 512)
        
        self.ln2 = torch.nn.LayerNorm(512)
        self.dp2 = torch.nn.Dropout(0.5)
        
        self.fc2 = torch.nn.Linear(512, classes)
        
        self.fc_aux = torch.nn.Linear(512, aux_classes)
        
        self.lnaux = torch.nn.LayerNorm(aux_classes)
        self.dpaux = torch.nn.Dropout(0.5)
                
        self.fc_gate = torch.nn.Linear(aux_classes, classes)
        self.act_gate = torch.nn.Sigmoid()

    def forward(self, x):
        x = self.ln1(x)
        x = self.dp1(x)
        
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        
        x = self.ln2(x)
        x = self.dp2(x)
        
#         aux
        aux = self.fc_aux(x)
#         aux = self.lnaux(aux)
#         aux = self.dpaux(aux)
        
#         linear bottom
        vqa = self.fc2(x)
        
        output = vqa * self.act_gate(self.fc_gate(aux))
        
        
        return output,aux

In [57]:
def run_model(model,dataloader,val_loader, optimizer,train = True ):
    if train:
        model.train()
    
    loss = nn.CrossEntropyLoss()
    
    total_loss = 0
    total_correct = 0
    total_samples = 0
    
    total_correct_ty = 0
    total_samples_ty = 0
    
    for (data, ans , ans_type) in tqdm(dataloader):
        data = data.to(device)
        ans = ans.to(device)
        ans_type = ans_type.to(device)
        optimizer.zero_grad()
        output , aux = model(data)
        
        loss_ans = loss(output, ans)
        loss_type = loss(aux,ans_type)
        loss_combined=loss_ans+loss_type
        total_loss += loss_combined.item()
        loss_combined.backward()
        optimizer.step()
        
        
#         Answer Accurracy
        _, predicted_labels = torch.max(output, dim=1)
        correct = (predicted_labels == ans).sum().item()
        total_correct += correct
        total_samples += ans.size(0)
        train_accuracy = total_correct / total_samples
        
#         Type Accuracy
        _, predicted_labels_ty = torch.max(aux, dim=1)
        correct_ty = (predicted_labels_ty == ans_type).sum().item()
        total_correct_ty += correct_ty
        total_samples_ty += ans_type.size(0)
        train_accuracy_ty = total_correct_ty / total_samples_ty
    
    total_train_accuracy = (train_accuracy + train_accuracy_ty) / 2
        
        
    if val_loader is not None:
        model.eval()

    # Initialize validation-specific variables
    val_loss = 0.0
    total_correct_val=0
    total_samples_val=0
    
    total_correct_val_ty=0
    total_samples_val_ty=0
    
    # Disable gradient calculation
    with torch.no_grad():
        # Iterate over the validation data
        for (data, ans , ans_type) in val_loader:
            data = data.to(device)
            ans = ans.to(device)
            ans_type = ans_type.to(device)
            
            # Forward pass
            output , aux = model(data)

            # Compute the loss
            loss_ans = loss(output, ans)
            loss_type = loss(aux,ans_type)
            loss_combined=loss_ans+loss_type
            val_loss += loss_combined.item()

            # Update validation metrics
            _, val_predicted = torch.max(output, dim=1)
            correct_val = (val_predicted == ans).sum().item()
            total_correct_val += correct_val
            total_samples_val += ans.size(0)
            val_accuracy = total_correct_val / total_samples_val
            
            _, val_predicted_ty = torch.max(aux, dim=1)
            correct_val_ty = (val_predicted_ty == ans_type).sum().item()
            total_correct_val_ty += correct_val_ty
            total_samples_val_ty += ans_type.size(0)
            val_accuracy_ty = total_correct_val_ty / total_samples_val_ty
        
        total_val_accuracy = (val_accuracy + val_accuracy_ty) / 2
        
    train_loss = total_loss/len(dataloader)
    val_loss = val_loss/len(val_loader)
    
    print(f"\nTrain Loss: {train_loss:.4f} | AVG Train ACC: {total_train_accuracy * 100:.4f}% | Val Loss: {val_loss:.4f} | AVG Val ACC: {total_val_accuracy * 100:.2f}%")
    
    print(f"\nTrain ANS ACC: {train_accuracy * 100:.4f}% | VAL ANS ACC: {val_accuracy * 100:.4f}% | Train TYPE ACC: {train_accuracy_ty * 100:.4f}% | VAL TYPE ACC: {val_accuracy_ty * 100:.2f}%\n")
        
    return train_loss, val_loss


In [58]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = Model()
model = model.to(device)
model = nn.DataParallel(model)

epoch = 125
training_loss = []
val_loss = []

optimizer = torch.optim.Adam(model.parameters(),1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=.1, threshold=1e-6)

early_stopping_patience = 15  # Number of epochs to wait for improvement
best_val_loss = float('inf')
epochs_without_improvement = 0

for e in range(epoch):
    print(f'Epoch: {e+1}', f'| LR: { optimizer.param_groups[0]["lr"] }')
    trLoss, vlLoss = run_model(model,train_dataloader,val_dataloader,optimizer)    
    training_loss.append(trLoss)
    val_loss.append(vlLoss)
    
    scheduler.step(vlLoss)
    
    # Check if validation loss has improved
    if vlLoss < best_val_loss:
        best_val_loss = vlLoss
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    # Check if early stopping criteria met
    if epochs_without_improvement >= early_stopping_patience:
        print(f"\nValidation loss hasn't improved for {early_stopping_patience} epochs. Early stopping.")
        break

Epoch: 1 | LR: 0.001


  0%|                                                                                                                                                                                                                           | 0/369 [00:00<?, ?it/s]


RuntimeError: Given normalized_shape=[3072], expected input with shape [*, 3072], but got input of size[64, 1, 1536]